# Detekcija in obravnava osamelcev — B: vaja (spletna trgovina)
**Akademija UI za poslovne aplikacije — Modul 1, popoldanski sklop**

Predavatelj je v zvezku `A_osamelci` postopek pokazal na telekomu.
Zdaj pregledate ekstremne vrednosti naročil spletne trgovine in predlagate obravnavo.

### Ponovitev
- **Osamelec**: vrednost, ki močno odstopa od ostalih. Lahko je
  - **napaka** — vrednost je nemogoča (vnos, prenos, napačna enota), ali
  - **veljaven ekstrem** — redka, a resnična vrednost.
- **Odkrivanje:** najprej graf (histogram, škatla z brki), nato **pravilo IQR**:
  kandidati so pod Q1 − 1,5 × IQR ali nad Q3 + 1,5 × IQR (= točke zunaj brkov).
- Pravilo IQR **ne pozna poslovanja** → dodamo **domenske omejitve** (npr. naročnina ≥ 0)
  in preverimo **povezane stolpce** iste stranke ali naročila.
- **Odzivi:** popravek iz preverjenega vira, označimo kot manjkajočo (`NaN`), odstranitev (z utemeljitvijo),
  omejevanje ali logaritem, ohranitev.
- **Osamelec ni nujno napaka; včasih je ravno najzanimivejši podatek.**

### Nastavitev okolja
Zaženite spodnjo celico (izberite jo in pritisnite **Shift + Enter**). Ničesar ne spreminjajte.
Celica deluje lokalno in v Google Colabu. Koda je skrita — odprete jo s klikom na celico.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:           # Google Colab: prenesemo gradivo z GitHuba
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not KOREN.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotnine>=0.15", "kagglehub"], check=True)
else:                                       # lokalno: poiščemo mapo, v kateri je utils/orodja.py
    KOREN = Path.cwd()
    while not (KOREN / "utils/orodja.py").exists() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
sys.path.insert(0, str(KOREN))              # da Python najde utils/orodja.py
print("Gradivo je v:", KOREN)

### Podatki
Funkcija `poskrbi_za_podatke()` iz `utils/orodja.py` ob prvem zagonu prenese podatke s Kaggla (~1 minuta)
in jih shrani v mapo `data/`. Kako to deluje, smo pokazali v zvezku `A_prvi_stik_s_podatki`.

In [ ]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

### Knjižnice in videz grafov
`pripravi_narocila` (zvezek `A_vizualizacija`), `histogram_in_skatla` (zvezek `A_vizualizacija`)
in `meje_iqr` (zvezek `A_osamelci`) uvozimo iz `utils/orodja.py`.

In [ ]:
import numpy as np
import pandas as pd
from plotnine import *

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, BARVE_ODHOD
from utils.orodja import pripravi_narocila, histogram_in_skatla, meje_iqr
nastavi_slog()      # enoten videz grafov (koda je v zvezku A_vizualizacija)

---
# Del B — Vaja: spletna trgovina
Ena vrstica = eno naročilo. Na vrhu celice zamenjate ime stolpca in celico ponovno zaženete (**Shift + Enter**).

In [ ]:
narocila = pripravi_narocila(DATA)
narocila.describe().round(1)

## B1. Poiščite kandidate
Poskusite: `"vrednost"` (z `LOG = True`), `"teza_g"`, `"postnina"`, `"dni_dostave"`, `"st_obrokov"`.
Pri `LOG = True` škatla računa brke na logaritmih, zato se ne ujemajo povsem z mejama IQR v izpisu.

**Vprašanje:** Koliko naročil označi pravilo IQR? So med njimi vrednosti 0? So to napake?

In [ ]:
STOLPEC = "vrednost"          # ← zamenjajte
LOG = True                    # ← logaritemska os (samo za prikaz)

s = narocila[STOLPEC].dropna().astype(float)
spodaj, zgoraj = meje_iqr(s)
oznaceni = (s < spodaj) | (s > zgoraj)
print(f"Meji IQR: od {spodaj:.1f} do {zgoraj:.1f}")
print(f"Označenih: {oznaceni.sum()} ({oznaceni.mean():.1%})")
print(f"Vrednosti = 0: {(s == 0).sum()}   vrednosti < 0: {(s < 0).sum()}")

histogram_in_skatla(narocila, STOLPEC, log=LOG)

## B2. Poglejte najbolj ekstremna naročila
Preverite **povezane stolpce**: koliko postavk ima naročilo, kakšna je poštnina, kategorija, ocena kupca …

In [ ]:
STOLPEC = "vrednost"          # ← zamenjajte

narocila.nlargest(8, STOLPEC)         # največje; za najmanjše: narocila.nsmallest(8, STOLPEC)

Ali so ekstremni časi dostave povezani z oceno kupca? `pd.cut` razdeli dneve v razrede, `groupby` izračuna povprečno oceno v vsakem.

In [ ]:
razredi = pd.cut(narocila["dni_dostave"], bins=[-1, 10, 20, 30, 60, 250],
                 labels=["do 10", "11–20", "21–30", "31–60", "nad 60"])
po_razredih = (narocila.groupby(razredi, observed=True)["ocena"]
               .agg(["mean", "count"]).reset_index())
print(po_razredih.round(2).to_string(index=False))

(ggplot(po_razredih, aes(x="dni_dostave", y="mean"))
 + geom_col(fill=MODRA)
 + labs(title="Povprečna ocena kupca glede na čas dostave", x="dni dostave", y="povprečna ocena (1–5)"))

## B3. Zapišite predlog
Dvakrat kliknite to celico in dopolnite za vsaj dva stolpca.

**Stolpec:** ...
**Napaka ali veljaven ekstrem? Zakaj?** ...
**Predlagana obravnava:** ...

**Za razpravo:** v teh podatkih negativnih vrednosti naročil ni. Če bi v stolpcu `vrednost` našli −120 BRL —
bi bila to napaka ali **vračilo**? Kako bi to preverili?

<details>
<summary><b>Rešitev (klikni)</b></summary>

Primer razmisleka (ne edini pravilni):

- **`vrednost`** — pravilo IQR označi ~8 % naročil (nad ~306 BRL). To so večinoma **veljavna** draga naročila
  (računalniki, telefonija, več postavk). Največje (13 440 BRL) ima 8 postavk, kategorija `fixed_telephony` — verjetno nakup podjetja.
  **Ohranimo.** Za model po potrebi `log1p`.
- **`teza_g`** — 22 naročil s težo **0 g**. Fizični izdelek ne more tehtati 0 g → napaka pri vnosu izdelka.
  **Označimo kot manjkajoče** (`mask`). Zelo težka naročila (100+ kg, pohištvo) so veljavna.
- **`postnina`** — 0 BRL pri ~340 naročilih. **Veljavno**: brezplačna dostava (akcija). Ohranimo.
- **`dni_dostave`** — do 209 dni. Možno (izgubljena pošiljka, zamuda), in poslovno **zelo zanimivo**: pri dostavi nad 30 dni
  je povprečna ocena ~2,2 (graf v B2). Ohranimo — to je ravno podatek, ki ga želimo videti.
- **`st_obrokov`** — 2 naročili z **0 obroki**. Plačilo ima vsaj en obrok → napaka. Označimo kot manjkajoče ali popravimo na 1.

**Negativna vrednost** je lahko napaka (minus pri vnosu) ali **vračilo / storno**. Preverimo v povezanih podatkih:
ali obstaja prvotno naročilo z enakim zneskom, kakšen je status naročila, kaj pravi računovodstvo.
Če gre za vračila, jih ne brišemo — za analizo prihodka jih potrebujemo.

</details>

---
### Za hitre
1. V telekomu ima ena stranka `stevilo_narocnin = 196`. Poglejte jo:
   `pd.read_csv(DATA / "telekom.csv").nlargest(3, "stevilo_narocnin")`. Napaka ali poslovni naročnik?
2. Koliko vsega prihodka trgovine prinesejo naročila, ki jih pravilo IQR označi pri `vrednost`?
   Namig: najprej `_, zgoraj = meje_iqr(narocila["vrednost"])`,
   nato `narocila.loc[narocila["vrednost"] > zgoraj, "vrednost"].sum() / narocila["vrednost"].sum()`